[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/39_linear_regression_solution.ipynb)

# 🟡 Solution: Linear Regression: Closed Form vs Gradient Descent

*Training · Medium*

Reference implementation. Try it yourself in `39_linear_regression.ipynb` first.

---
Fit a linear model $\hat y = Xw + b$ two ways and return `(w, b)`.

Both methods minimise the same ridge objective (note that the bias is **not**
penalised):

$$J(w, b) = \frac{1}{N}\Bigl( \lVert Xw + b\mathbf{1} - y \rVert_2^2 + \lambda \lVert w \rVert_2^2 \Bigr)$$

Writing $\tilde X = [\,X \mid \mathbf{1}\,]$ and $\theta = [w; b]$, the stationary
point is the **normal equations**

$$(\tilde X^\top \tilde X + \lambda M)\,\theta = \tilde X^\top y, \qquad M = \mathrm{diag}(1,\dots,1,0)$$

### Rules
- Signature: `linear_regression(X, y, *, l2=0.0, method="closed_form", lr=0.1, steps=200)`
- `X` is `(N, D)`, `y` is `(N,)`; return `(w, b)` with `w` of shape `(D,)` and `b` a **scalar**
- Banned: `jnp.linalg.inv`, `jnp.linalg.pinv`, `optax`, `flax` optimizers
- `method="closed_form"`: solve with `jnp.linalg.lstsq` (or an explicit `jnp.linalg.qr`).
  **Do not form $\tilde X^\top \tilde X$** — see below. Handle a rank-deficient `X` at
  $\lambda = 0$ without producing `nan`
- `method="gd"`: full-batch gradient descent on $J$, starting from $\theta = 0$, for
  exactly `steps` iterations with step size `lr`. Use `jax.lax.fori_loop` — a Python
  loop unrolls into `steps` copies of the graph and blows up compile time
- Both paths must run under `jit`, and `vmap` over a stack of datasets

### Why lstsq beats inverting $\tilde X^\top \tilde X$
Forming the Gram matrix **squares the condition number**:
$\kappa(\tilde X^\top \tilde X) = \kappa(\tilde X)^2$. The relative error of a
solve is roughly $\kappa \cdot \epsilon_{\text{machine}}$, and float32 has
$\epsilon \approx 1.2 \times 10^{-7}$ — about 7 decimal digits.

Take a degree-6 polynomial design on $t \in [0, 1]$, which is nothing exotic:
$\kappa(\tilde X) \approx 2 \times 10^4$, so $\kappa(\tilde X^\top \tilde X)
\approx 4 \times 10^8$. Every digit is gone. Run it and the normal equations
return coefficients wrong in the *first* digit — an absolute error of order 1
(about 8 on the machine this was written on; once $\kappa \epsilon > 1$ the
answer is noise, so the exact figure is not reproducible), against
$\sim 4\times 10^{-5}$ from `lstsq` on identical inputs. QR factorises
$\tilde X$ directly and never squares anything, so its error tracks
$\kappa(\tilde X)$ — you keep half the digits the normal equations throw away.

This is also the real reason ridge "stabilises" the fit. Adding $\lambda$ shifts
every squared singular value up, so the effective condition number becomes
$(\sigma_{\max}^2 + \lambda)/(\sigma_{\min}^2 + \lambda)$. At $\lambda = 0$ with
duplicated (collinear) columns the Gram matrix is exactly singular and a solve
returns `nan`; `lstsq` instead returns the **minimum-norm** solution, which fits
just as well and splits the weight evenly between the collinear features.

### So why would you ever iterate?
The closed form costs $O(ND^2 + D^3)$ and needs all $N$ rows resident. Gradient
descent costs $O(ND)$ per step, streams minibatches, and never materialises
anything of size $D \times D$ — at $D = 10^6$ (an embedding table) the Gram
matrix alone would be $10^{12}$ entries. And the moment the model stops being
linear in its parameters, there is no closed form at all. Linear regression is
the one place you can check your optimizer against ground truth, which is
exactly why interviewers ask for both and then ask which one they should ship.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def linear_regression(X, y, *, l2=0.0, method="closed_form", lr=0.1, steps=200):
    X = jnp.asarray(X)
    y = jnp.asarray(y)
    n, d = X.shape
    # Absorb the bias as a constant feature: theta = [w; b].
    Xb = jnp.concatenate([X, jnp.ones((n, 1), X.dtype)], axis=1)

    if method == "closed_form":
        # Ridge by data augmentation: the extra rows add sqrt(l2)*w to the
        # residual, i.e. l2*||w||^2 to the objective. The trailing zero column
        # is what keeps the bias out of the penalty.
        reg = jnp.sqrt(jnp.asarray(l2, Xb.dtype)) * jnp.concatenate(
            [jnp.eye(d, dtype=Xb.dtype), jnp.zeros((d, 1), Xb.dtype)], axis=1
        )
        Xa = jnp.concatenate([Xb, reg], axis=0)
        ya = jnp.concatenate([y, jnp.zeros((d,), y.dtype)])
        # lstsq factorises Xa itself — it never forms Xa.T @ Xa.
        theta = jnp.linalg.lstsq(Xa, ya)[0]

    elif method == "gd":
        mask = jnp.concatenate(
            [jnp.ones((d,), Xb.dtype), jnp.zeros((1,), Xb.dtype)]
        )

        def body(_, theta):
            resid = Xb @ theta - y
            grad = 2.0 / n * (Xb.T @ resid + l2 * mask * theta)
            return theta - lr * grad

        # fori_loop compiles ONE body and loops it; a Python for would unroll.
        theta = jax.lax.fori_loop(0, steps, body, jnp.zeros((d + 1,), Xb.dtype))

    else:
        raise ValueError(f"unknown method {method!r}")

    return theta[:d], theta[d]

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

# The conditioning blow-up, on a degree-6 polynomial design.
t = jnp.linspace(0.0, 1.0, 60)
X = jnp.stack([t ** k for k in range(1, 7)], axis=1)
true_w = jnp.array([1.0, -2.0, 3.0, -1.0, 0.5, 2.0])
y = X @ true_w + 0.3

Xb = jnp.concatenate([X, jnp.ones((60, 1))], axis=1)
print("cond(X~)     =", float(jnp.linalg.cond(Xb)))
# ~2.6e8 rather than the exact 4.0e8: estimating the condition number of the
# Gram matrix in float32 is itself past the point where float32 can answer.
print("cond(X~^T X~)=", float(jnp.linalg.cond(Xb.T @ Xb)))

w_ls, b_ls = linear_regression(X, y)
theta_ne = jnp.linalg.solve(Xb.T @ Xb, Xb.T @ y)      # the tempting one-liner
print("lstsq  max coef error:", float(jnp.max(jnp.abs(w_ls - true_w))))
print("normal max coef error:", float(jnp.max(jnp.abs(theta_ne[:6] - true_w))))

# Gradient descent lands in the same place on a well-conditioned problem.
import jax
Xg = jax.random.normal(jax.random.key(0), (200, 3))
yg = Xg @ jnp.array([2.0, -1.0, 0.5]) + 3.0
print("closed form:", linear_regression(Xg, yg))
print("gd  2000 it:", linear_regression(Xg, yg, method="gd", lr=0.1, steps=2000))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("linear_regression")